# KLTN — ABSA Token Merging · SMOKE TEST

Kiểm tra **toàn bộ đường ống chạy được** trên một dataset tí hon, trước khi tốn
hàng chục giờ GPU cho lượt thật.

> Muốn chạy thật trên dữ liệu đầy đủ → dùng **`kaggle_full_run.ipynb`**.

## `--smoke` làm gì

| | |
|---|---|
| Dữ liệu | `dataset_smoke/` — 72 train / 32 dev / 32 test, **phủ đủ 6 category × 3 sentiment** |
| Epoch | 1 |
| Seed | 1 (`42`) |
| Biến thể | 3 — `lcf_scm_cdm` (post-ToMe resize), `lcf_scm_cdm_compact` (compact), `lcf_pre_scm` (pre-ToMe) |
| Output | **`smoke_run/`** — không đụng vào kết quả thật |
| Thời gian | ~5–10 phút trên GPU (phần lớn là tải `t5-base` + `bert-base-uncased`) |

Mẫu được chọn phủ nhãn chứ không phải N mẫu đầu: dataset sắp xếp theo category
nên 48 mẫu đầu chỉ có `SERVICE/Positive` — một lớp duy nhất, và
`classification_report(target_names=...)` trong `run_joint_experiments.py` sẽ ném
`ValueError`.

## Trước khi chạy

| Mục | Giá trị |
|---|---|
| **Accelerator** | `GPU T4 x2` hoặc `GPU P100` |
| **Internet** | `On` (bắt buộc — `git clone` + tải model HuggingFace) |


## 1 · Cấu hình

Bật sẵn trong panel bên phải: **Accelerator = GPU**, **Internet = On**.


In [ ]:
import os
from pathlib import Path

# ── Nguồn code ────────────────────────────────────────────────────────────
REPO_URL  = "https://github.com/hotuyen21pt/KLTN-Token-Merging.git"
BRANCH    = "tuyen"
REPO_NAME = "KLTN-Token-Merging"

# Repo private? Thêm Kaggle Secret tên GITHUB_TOKEN (Add-ons → Secrets).
USE_TOKEN = False

# ── Thư mục làm việc ──────────────────────────────────────────────────────
# /kaggle/working được commit lại sau phiên (~20 GB).
# /kaggle/temp bị xoá hết phiên → để cache model, khỏi tốn quota output.
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO_DIR  = WORK_ROOT / REPO_NAME
CACHE_DIR = Path("/kaggle/temp/hf") if Path("/kaggle").exists() else WORK_ROOT / ".hf"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["MPLBACKEND"] = "Agg"   # Kaggle headless

print(f"Repo dir : {REPO_DIR}")
print(f"HF cache : {CACHE_DIR}")
print(f"Branch   : {BRANCH}")


## 2 · Clone / pull code từ GitHub

Chạy lại được nhiều lần:

- **Lần đầu** → `git clone --branch <BRANCH> --single-branch`
- **Lần sau** → `remote set-url` → `fetch --all --prune --tags` →
  `checkout -B` → `reset --hard origin/<BRANCH>` → `clean -fd`

`reset --hard` đảm bảo code khớp *chính xác* GitHub. Kết quả thực nghiệm nằm
ngoài vùng git theo dõi (đã `.gitignore`) nên **không** bị xoá.


In [ ]:
import subprocess, sys, shutil

def sh(cmd, cwd=None, check=True):
    """Chạy lệnh, in output theo thời gian thực (train chạy nhiều giờ)."""
    cmd = [str(c) for c in cmd]
    print(f"$ {' '.join(cmd)}", flush=True)
    proc = subprocess.Popen(
        cmd, cwd=cwd, text=True, encoding="utf-8", errors="replace",
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if check and proc.returncode != 0:
        raise RuntimeError(f"Lệnh thất bại (exit {proc.returncode}): {' '.join(cmd)}")
    return proc.returncode


clone_url = REPO_URL
if USE_TOKEN:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    clone_url = REPO_URL.replace("https://", f"https://{token}@")
    print("Dùng GITHUB_TOKEN từ Kaggle Secrets")

if (REPO_DIR / ".git").is_dir():
    print(f"Đã có repo tại {REPO_DIR} → cập nhật\n")
    sh(["git", "remote", "set-url", "origin", clone_url], cwd=REPO_DIR)
    sh(["git", "fetch", "--all", "--prune", "--tags"], cwd=REPO_DIR)
    sh(["git", "checkout", "-B", BRANCH, f"origin/{BRANCH}"], cwd=REPO_DIR)
    sh(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)
    sh(["git", "clean", "-fd"], cwd=REPO_DIR)
else:
    print(f"Chưa có repo → clone nhánh {BRANCH}\n")
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    sh(["git", "clone", "--branch", BRANCH, "--single-branch", clone_url, str(REPO_DIR)])

if USE_TOKEN:   # giấu token khỏi .git/config
    sh(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_DIR)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print()
sh(["git", "log", "--oneline", "-5"], cwd=REPO_DIR)
sh(["git", "status", "--short", "--branch"], cwd=REPO_DIR)
print(f"\nThư mục hiện tại: {os.getcwd()}")


## 3 · Cài thư viện

Image Kaggle đã có `torch`, `transformers`, `scikit-learn`, `matplotlib`, `pandas`.
Cell này bù những gói còn thiếu.

**`Levenshtein` là bắt buộc** — `src/normalization.py` import nó, thiếu là stage
`ate` / `ate_infer` / `gas` chết ngay với `ModuleNotFoundError`. Cell cài **từng
gói một lệnh pip riêng** (gộp chung thì một gói resolve hỏng làm pip bỏ cả lô)
và **dừng hẳn** nếu gói bắt buộc vẫn thiếu, thay vì chỉ cảnh báo rồi đi tiếp.

`pyabsa` là tuỳ chọn — không có thì các stage chính vẫn chạy.


In [ ]:
FULL_INSTALL = False   # True = cài đúng requirements.txt (lâu, dễ đụng torch của Kaggle)

# (tên pip, tên import, bắt buộc?)
PACKAGES = [
    ("python-Levenshtein>=0.25.0", "Levenshtein", True),
    ("seaborn",                    "seaborn",     True),
    ("tqdm",                       "tqdm",        True),
    ("pyabsa>=2.4.0,<3",           "pyabsa",      False),
]

if FULL_INSTALL:
    sh([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
else:
    for spec, _mod, required in PACKAGES:
        rc = sh([sys.executable, "-m", "pip", "install", "-q", spec], check=False)
        if rc != 0:
            note = "  (BẮT BUỘC)" if required else "  (tuỳ chọn, bỏ qua được)"
            print(f"  [!] pip install {spec} → exit {rc}{note}")

print("\n" + "=" * 60)
missing = []
CHECK = PACKAGES + [
    ("torch", "torch", True), ("transformers", "transformers", True),
    ("scikit-learn", "sklearn", True), ("matplotlib", "matplotlib", True),
    ("pandas", "pandas", True),
]
for spec, mod, required in CHECK:
    try:
        m = __import__(mod)
        print(f"  {mod:<14}: {getattr(m, '__version__', 'ok')}")
    except Exception as e:
        print(f"  {mod:<14}: {'THIẾU (BẮT BUỘC)' if required else 'thiếu (tuỳ chọn)'}"
              f" — {type(e).__name__}")
        if required:
            missing.append(spec)
print("=" * 60)

if missing:
    raise SystemExit(
        "DỪNG — thiếu gói bắt buộc.\nCài tay rồi RESTART kernel:\n"
        + "\n".join(f"  !pip install {s}" for s in missing)
    )
print("Đủ thư viện bắt buộc.")


## 4 · Kiểm tra môi trường


In [ ]:
sh(["nvidia-smi"], check=False)
print()
sh([sys.executable, "run_all.py", "--list"])


## 5 · Chạy smoke test

Bỏ qua `uos` vì Kaggle không có Ollama.


In [ ]:
rc = sh([sys.executable, "run_all.py", "--smoke", "--skip", "uos"], check=False)

print("\n" + "=" * 70)
print("SMOKE TEST: " + ("ĐẠT — đường ống chạy được" if rc == 0
                        else f"HỎNG (exit {rc}) — xem cell chẩn đoán bên dưới"))
print("=" * 70)


## 6 · Bảng trạng thái từng stage


In [ ]:
import json

SMOKE_DIR = REPO_DIR / "smoke_run"
status_file = SMOKE_DIR / "reports" / "run_status.json"

failed = []
if status_file.is_file():
    data = json.loads(status_file.read_text(encoding="utf-8"))
    print(f"{'stage':<12} {'trạng thái':<10} {'phút':>6}  ghi chú")
    print("-" * 90)
    for s in data["stages"]:
        print(f"{s['stage']:<12} {s['status']:<10} "
              f"{s.get('duration_sec', 0) / 60:6.1f}  {s.get('note', '')[:58]}")
        if s["status"] != "OK":
            failed.append(s["stage"])
    print("-" * 90)
    print(f"{len(data['stages']) - len(failed)}/{len(data['stages'])} stage OK")
else:
    print("Chưa có smoke_run/reports/run_status.json — chạy cell 5 trước.")


## 7 · Chẩn đoán stage hỏng

In 40 dòng cuối log của mỗi stage lỗi. Dòng `Traceback` / `Error` đầu tiên
thường là nguyên nhân gốc — các stage hỏng sau đó phần lớn chỉ là đổ dây chuyền
(ví dụ `ate` hỏng → không có checkpoint → `ate_infer`, `triplet` hỏng theo).


In [ ]:
if not failed:
    print("Không có stage nào hỏng.")
else:
    log_dir = SMOKE_DIR / "reports" / "logs"
    for stage in failed:
        matches = sorted(log_dir.glob(f"{stage}*.log"))
        if not matches:
            print(f"\n### {stage}: không có file log")
            continue
        for lg in matches:
            lines = lg.read_text(encoding="utf-8", errors="replace").splitlines()
            print(f"\n{'=' * 78}")
            print(f"### {lg.name}  ({len(lines)} dòng) — 40 dòng cuối")
            print("=" * 78)
            for l in lines[-40:]:
                print(l)

    # Tóm tắt các lỗi Python gặp phải
    print(f"\n{'=' * 78}")
    print("### Lỗi tìm thấy trong log")
    print("=" * 78)
    seen = set()
    for lg in sorted(log_dir.glob("*.log")):
        for l in lg.read_text(encoding="utf-8", errors="replace").splitlines():
            s = l.strip()
            if any(s.startswith(k) for k in
                   ("ModuleNotFoundError", "ImportError", "ValueError",
                    "FileNotFoundError", "RuntimeError", "KeyError", "OSError")):
                if s not in seen:
                    seen.add(s)
                    print(f"  [{lg.stem}] {s[:110]}")
    if not seen:
        print("  (không thấy dòng exception nào)")


## 8 · Báo cáo của lượt smoke

Các con số ở đây **không có giá trị khoa học** — 1 epoch trên 72 mẫu. Chỉ dùng để
xác nhận mọi stage chạy và ghi được file.


In [ ]:
from IPython.display import Markdown, display

report = SMOKE_DIR / "reports" / "REPORT.md"
if report.is_file():
    display(Markdown(report.read_text(encoding="utf-8")))
else:
    print(f"Chưa có {report}")


## 9 · Bước tiếp theo

- **Smoke ĐẠT** → mở `kaggle_full_run.ipynb` để chạy dữ liệu đầy đủ
- **Smoke HỎNG** → sửa theo cell 7, `Run All` lại notebook này

Dọn sandbox nếu muốn (không bắt buộc — đã nằm trong `.gitignore`):


In [ ]:
# shutil.rmtree(REPO_DIR / "smoke_run", ignore_errors=True)
# shutil.rmtree(REPO_DIR / "dataset_smoke", ignore_errors=True)
# print("Đã dọn sandbox.")
print("Bỏ comment 2 dòng trên nếu muốn dọn smoke_run/ và dataset_smoke/.")
